In [ ]:
# ── Setup Google Colab ───────────────────────────────────────────────────────
import os, sys

# Montar Google Drive (descomenta si guardás el CSV ahí)
# from google.colab import drive
# drive.mount("/content/drive")

# Subir CSV desde tu máquina local
from google.colab import files
uploaded = files.upload()   # selecciona tweets_clasificados.csv

import torch
device = 0 if torch.cuda.is_available() else -1
device_name = torch.cuda.get_device_name(0) if device == 0 else "CPU"
print(f"Dispositivo: {device_name}")

# Ajustar INPUT_CSV al nombre del archivo subido
INPUT_CSV = list(uploaded.keys())[0]
print(f"Archivo cargado: {INPUT_CSV}")


# Benchmark simple de modelos de odio en español

Esta notebook toma un CSV local de 100 casos y aplica varios clasificadores de odio en español sobre el mismo conjunto, para que puedas comparar salidas.

Incluye:
- modelos de Hugging Face con salida binaria o fácilmente binarizable
- `pysentimiento` para detección de odio en español
- guardado de resultados por fila
- una comparación simple entre modelos
- una columna opcional de etiqueta de referencia si quieres contrastar manualmente

## 1. Instalación

Descomenta si hace falta.

In [1]:
!pip install -q pandas transformers torch pysentimiento tqdm scikit-learn accelerate



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports

In [1]:
import json
import re
from typing import Dict, Any, Optional

import pandas as pd
from tqdm.auto import tqdm

from transformers import pipeline
from pysentimiento import create_analyzer

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## 3. Configuración

Ajusta estas variables según tu archivo.

In [2]:
INPUT_CSV = 'DNU_Data/tweets_clasificados.csv'
OUTPUT_CSV = 'outputs/DNU_modelos.csv'

TEXT_COLUMN = None
GOLD_COLUMN = None
MAX_ROWS = None

## 4. Cargar datos

In [3]:
df = pd.read_csv(INPUT_CSV)
print(f'Filas originales: {len(df)}')
print('Columnas disponibles:')
print(list(df.columns))
df.head(3)

Filas originales: 259752
Columnas disponibles:
['created_at', 'id', 'id_str', 'text', 'source', 'truncated', 'in_reply_to_status_id', 'in_reply_to_status_id_str', 'in_reply_to_user_id', 'quoted_status.place.place_type', 'quoted_status.place.name', 'retweeted_status.quoted_status.quoted_status_id', 'is_retweet', 'caracteres', 'text_pp', 'tokens', 'fecha', 'fecha_dia', 'CALLS', 'WOMEN', 'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL', 'labels', 'n_labels']


,created_at,id,id_str,text,source,truncated,in_reply_to_status_id,in_reply_to_status_id_str,in_reply_to_user_id,quoted_status.place.place_type,...,WOMEN,LGBTI,RACISM,CLASS,POLITICS,DISABLED,APPEARANCE,CRIMINAL,labels,n_labels
0,Fri Mar 05 11:01:46 +0000 2021,1367792447334580225,1367792447334580225,RT @CELS_Argentina: ✅Celebramos la decisión de...,"<a href=""http://twitter.com/download/android"" ...",False,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,[],0
1,Fri Mar 05 11:06:31 +0000 2021,1367793643239702529,1367793643239702529,RT @LANACION: Ley de Migraciones: el Gobierno ...,"<a href=""http://twitter.com/download/android"" ...",False,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,[],0
2,Fri Mar 05 11:06:33 +0000 2021,1367793651192070144,1367793651192070144,"RT @CRLDEMONIO: Y lo peor , el alto mando d...","<a href=""http://twitter.com/download/android"" ...",False,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,[],0


In [4]:
text_col = 'text'
work_df = df.copy().head(MAX_ROWS).reset_index(drop=True)
print('Columna de texto:', text_col)
print('Filas a procesar:', len(work_df))

Columna de texto: text
Filas a procesar: 259752


## 5. Modelos a comparar

Estos son los modelos incluidos por defecto.

- `pysentimiento` usa RoBERTuito y ofrece detección de odio para español.
- `cardiffnlp/twitter-xlm-roberta-base-hate-spanish` está orientado a hate speech en español sobre Twitter/social media.
- `JonatanGk/roberta-base-bne-finetuned-hate-speech-offensive-spanish` es un clasificador español de hate/offensive.
- `delarosajav95/HateSpeech-BETO-cased-v2` está pensado para detección de odio en español con cobertura de varias formas de discriminación.

Si alguno da problemas de descarga o de formato de etiquetas, puedes desactivarlo.

In [5]:
USE_MODELS = {
    'pysentimiento_hate': True,
    'cardiff_xlm_hate_es': True,
    'bne_hate_offensive_es': True,
    'beto_hate_v2_es': True,
}

HF_MODEL_NAMES = {
    'cardiff_xlm_hate_es': 'cardiffnlp/twitter-xlm-roberta-base-hate-spanish',
    'bne_hate_offensive_es': 'JonatanGk/roberta-base-bne-finetuned-hate-speech-offensive-spanish',
    'beto_hate_v2_es': 'delarosajav95/HateSpeech-BETO-cased-v2',
}

## 6. Cargar pipelines

In [6]:
loaded_models: Dict[str, Any] = {}
load_errors: Dict[str, str] = {}

# device ya definido en la celda de setup (0 = GPU, -1 = CPU)

if USE_MODELS.get("pysentimiento_hate"):
    try:
        loaded_models["pysentimiento_hate"] = create_analyzer(task="hate_speech", lang="es")
        print("OK pysentimiento_hate")
    except Exception as e:
        load_errors["pysentimiento_hate"] = str(e)
        print("ERROR pysentimiento_hate:", e)

for key, model_name in HF_MODEL_NAMES.items():
    if USE_MODELS.get(key):
        try:
            loaded_models[key] = pipeline(
                "text-classification",
                model=model_name,
                tokenizer=model_name,
                truncation=True,
                device=device
            )
            print(f"OK {key}: {model_name}")
        except Exception as e:
            load_errors[key] = str(e)
            print(f"ERROR {key}:", e)

print("
Errores de carga:")
load_errors


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

OK pysentimiento_hate


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

OK cardiff_xlm_hate_es: cardiffnlp/twitter-xlm-roberta-base-hate-spanish


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

OK bne_hate_offensive_es: JonatanGk/roberta-base-bne-finetuned-hate-speech-offensive-spanish


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

OK beto_hate_v2_es: delarosajav95/HateSpeech-BETO-cased-v2

Errores de carga:


{}

## 7. Funciones auxiliares

Como cada modelo devuelve etiquetas distintas, las convertimos a un esquema binario común:

- `1` = odio / ofensivo / hateful
- `0` = no odio / no ofensivo / non-hate

También guardamos la etiqueta cruda original para inspección.

In [7]:
NEGATIVE_TOKENS = {
    'non_hateful','non-hateful','not_hate','not-hate','no_hate','no-hate',
    'normal','neutral','none','clean','non-offensive','not offensive',
    'label_0','0',
    'neither',       # bne_hate_offensive_es: ni odio ni ofensivo
    'not-hate',
}
POSITIVE_HINTS = {'hate','hateful','offensive','toxic','abusive','hs','aggressive','label_1','1'}

def normalize_label_text(x: Any) -> str:
    if x is None:
        return ''
    s = str(x).strip().lower()
    s = re.sub(r'\s+', ' ', s)
    return s

def binary_from_label(label: Any, score: Optional[float] = None) -> Optional[int]:
    s = normalize_label_text(label)
    if s in NEGATIVE_TOKENS:
        return 0
    for token in POSITIVE_HINTS:
        if token in s and s not in NEGATIVE_TOKENS:
            return 1
    return None

def run_pysentimiento(text: str) -> Dict[str, Any]:
    analyzer = loaded_models['pysentimiento_hate']
    pred = analyzer.predict(text)
    output_label = getattr(pred, 'output', None)
    probas = getattr(pred, 'probas', None)

    # output es una lista de categorías detectadas; lista vacía = sin odio
    if isinstance(output_label, list):
        binary = 1 if len(output_label) > 0 else 0
    elif output_label is not None:
        s = normalize_label_text(output_label)
        if s in {'hateful', 'hate'}:
            binary = 1
        elif s in {'non-hateful', 'non_hateful', 'non-hate', 'not-hate'}:
            binary = 0
        else:
            binary = None
    else:
        binary = None

    return {'raw_label': output_label, 'score': None, 'binary_pred': binary, 'raw_probas': str(probas) if probas is not None else None}

def run_hf_pipeline(model_key: str, text: str) -> Dict[str, Any]:
    clf = loaded_models[model_key]
    pred = clf(text)
    if isinstance(pred, list) and len(pred) > 0 and isinstance(pred[0], dict):
        best = pred[0]
        label = best.get('label')
        score = best.get('score')
    else:
        label = str(pred)
        score = None
    binary = binary_from_label(label, score)
    return {'raw_label': label, 'score': score, 'binary_pred': binary, 'raw_probas': None}


## 8. Prueba rápida con la primera fila

In [8]:
sample_text = str(work_df[text_col].iloc[0])
print(sample_text)

test_outputs = {}
for model_key in loaded_models.keys():
    try:
        if model_key == 'pysentimiento_hate':
            test_outputs[model_key] = run_pysentimiento(sample_text)
        else:
            test_outputs[model_key] = run_hf_pipeline(model_key, sample_text)
    except Exception as e:
        test_outputs[model_key] = {'error': str(e)}

test_outputs

RT @CELS_Argentina: ✅Celebramos la decisión del Poder Ejecutivo de derogar el DNU 70/2017, cuya implementación significó un claro retroceso…


{'pysentimiento_hate': {'raw_label': [],
  'score': None,
  'binary_pred': 0,
  'raw_probas': "{'hateful': 0.010241943411529064, 'targeted': 0.00691426545381546, 'aggressive': 0.007565696723759174}"},
 'cardiff_xlm_hate_es': {'raw_label': 'NOT-HATE',
  'score': 0.9847297072410583,
  'binary_pred': 0,
  'raw_probas': None},
 'bne_hate_offensive_es': {'raw_label': 'NEITHER',
  'score': 0.8745400905609131,
  'binary_pred': 0,
  'raw_probas': None},
 'beto_hate_v2_es': {'raw_label': 'LABEL_0',
  'score': 0.9998912811279297,
  'binary_pred': 0,
  'raw_probas': None}}

## 9. Correr todos los modelos sobre las 100 filas

In [9]:
BATCH_SIZE = 64        # bajar a 32 si hay OOM en GPU
CHECKPOINT_PATH = OUTPUT_CSV.replace(".csv", "_checkpoint.csv")

texts = work_df[text_col].fillna("").tolist()
n = len(texts)

# Inicializar columnas de resultados
for model_key in loaded_models.keys():
    for suffix in ["raw_label", "score", "binary_pred", "raw_probas", "error"]:
        work_df[f"{model_key}__{suffix}"] = None

# ── pysentimiento: batch nativo ──────────────────────────────────────────────
if "pysentimiento_hate" in loaded_models:
    analyzer = loaded_models["pysentimiento_hate"]
    all_preds = []
    for i in tqdm(range(0, n, BATCH_SIZE), desc="pysentimiento_hate"):
        batch = texts[i:i + BATCH_SIZE]
        preds = analyzer.predict(batch)
        all_preds.extend(preds)

    for i, pred in enumerate(all_preds):
        output_label = getattr(pred, "output", None)
        probas = getattr(pred, "probas", None)
        if isinstance(output_label, list):
            binary = 1 if len(output_label) > 0 else 0
        else:
            s = normalize_label_text(output_label)
            binary = 1 if s in {"hateful","hate"} else (0 if s in {"non-hateful","non_hateful","non-hate","not-hate"} else None)
        work_df.at[i, "pysentimiento_hate__raw_label"] = str(output_label)
        work_df.at[i, "pysentimiento_hate__binary_pred"] = binary
        work_df.at[i, "pysentimiento_hate__raw_probas"] = str(probas) if probas is not None else None

# ── Modelos HuggingFace: batching via pipeline() ─────────────────────────────
for model_key, clf in loaded_models.items():
    if model_key == "pysentimiento_hate":
        continue

    raw_labels, scores, binary_preds = [], [], []

    for out in tqdm(clf(texts, batch_size=BATCH_SIZE), total=n, desc=model_key):
        label = out.get("label")
        score = out.get("score")
        raw_labels.append(label)
        scores.append(score)
        binary_preds.append(binary_from_label(label, score))

    work_df[f"{model_key}__raw_label"] = raw_labels
    work_df[f"{model_key}__score"] = scores
    work_df[f"{model_key}__binary_pred"] = binary_preds

# Guardar resultado final
work_df.to_csv(OUTPUT_CSV, index=False)
print(f"Resultados guardados en {OUTPUT_CSV}")


## 10. Vista rápida de resultados

In [11]:
cols_to_show = [text_col]
if GOLD_COLUMN is not None and GOLD_COLUMN in work_df.columns:
    cols_to_show.append(GOLD_COLUMN)
for model_key in loaded_models.keys():
    cols_to_show.extend([f'{model_key}__raw_label', f'{model_key}__binary_pred'])
work_df[cols_to_show].head(10)

,text,pysentimiento_hate__raw_label,pysentimiento_hate__binary_pred,cardiff_xlm_hate_es__raw_label,cardiff_xlm_hate_es__binary_pred,bne_hate_offensive_es__raw_label,bne_hate_offensive_es__binary_pred,beto_hate_v2_es__raw_label,beto_hate_v2_es__binary_pred
0,Extranjeros participan de los destrozos realiz...,[],0,NOT-HATE,0,NEITHER,0,LABEL_0,0
1,@pepegoyomr @CARLOSLGUERRERO Al contrario!!! E...,[],0,NOT-HATE,0,OFFENSIVE,1,LABEL_0,0
2,La diferencia con los kirchneristas ya no es s...,[],0,NOT-HATE,0,NEITHER,0,LABEL_0,0
3,@Lady_Chiyome La dictadura de pelusones del 73...,[],0,NOT-HATE,0,NEITHER,0,LABEL_0,0
4,@Mari21Llll La pinché Momia solo es un florero...,"[hateful, targeted, aggressive]",1,HATE,1,NEITHER,0,LABEL_0,0
5,El Gobierno derogó el decreto de Mauricio Macr...,[],0,NOT-HATE,0,NEITHER,0,LABEL_0,0
6,@MikeHam060 Fueron expulsados por tener antece...,[],0,NOT-HATE,0,NEITHER,0,LABEL_0,0
7,@bastogne1943 Exacto... Hoy los únicos capaces...,[hateful],1,NOT-HATE,0,NEITHER,0,LABEL_1,1
8,@rubenprofe @Elputonisa @ArielSanche2 @Dasmatn...,[],0,NOT-HATE,0,NEITHER,0,LABEL_0,0
9,@neuquen24horas @alferdez @alferdezprensa siem...,[],0,NOT-HATE,0,NEITHER,0,LABEL_0,0


## 11. Comparación entre modelos

Primero vemos cuántos casos positivos marca cada uno.

In [12]:
summary_rows = []
for model_key in loaded_models.keys():
    pred_col = f'{model_key}__binary_pred'
    valid = pd.to_numeric(work_df[pred_col], errors='coerce')
    summary_rows.append({'model': model_key, 'n_valid_preds': int(valid.notna().sum()), 'n_pred_hate': int((valid == 1).sum()), 'hate_rate': float((valid == 1).mean()) if valid.notna().any() else None})
summary_df = pd.DataFrame(summary_rows).sort_values('hate_rate', ascending=False)
summary_df

,model,n_valid_preds,n_pred_hate,hate_rate
2,bne_hate_offensive_es,100,42,0.42
0,pysentimiento_hate,100,26,0.26
1,cardiff_xlm_hate_es,100,13,0.13
3,beto_hate_v2_es,100,7,0.07


## 12. Acuerdo entre modelos

Esto permite ver si dos modelos tienden a coincidir.

In [13]:
pred_cols = [f'{m}__binary_pred' for m in loaded_models.keys()]
pred_matrix = work_df[pred_cols].apply(pd.to_numeric, errors='coerce')
agreement = pred_matrix.corr()
agreement

,pysentimiento_hate__binary_pred,cardiff_xlm_hate_es__binary_pred,bne_hate_offensive_es__binary_pred,beto_hate_v2_es__binary_pred
pysentimiento_hate__binary_pred,1.000000,0.584351,0.142269,0.284141
cardiff_xlm_hate_es__binary_pred,0.584351,1.000000,0.153025,0.127030
bne_hate_offensive_es__binary_pred,0.142269,0.153025,1.000000,0.084174
beto_hate_v2_es__binary_pred,0.284141,0.127030,0.084174,1.000000


## 13. Comparación contra una etiqueta de referencia (opcional)

Si defines `GOLD_COLUMN`, esta celda intenta convertirla a binaria y calcula métricas.

In [21]:
# ── 1. Preparar gold desde df_check (ya tiene HATEFUL garantizado) ───────────
df_gold = df_check[['id', 'text', 'HATEFUL']].rename(columns={'text': 'text_original'})

# ── 2. Merge con work_df — excluir HATEFUL y text de work_df para evitar colisiones
cols_work = [c for c in work_df.columns if c not in ('HATEFUL', 'text')]
results_df = (
    work_df[cols_work + ['text']].rename(columns={'text': 'text_work'})
    .merge(df_gold, on='id', how='left')
)

print("Verificación del merge (primeras 5 filas):")
display(results_df[['id', 'text_work', 'text_original', 'HATEFUL']].head())

# ── 3. Benchmark contra HATEFUL (gold) ──────────────────────────────────────
gold_bin = results_df['HATEFUL'].astype(int)

for model_key in loaded_models.keys():
    pred_col = f'{model_key}__binary_pred'
    preds = pd.to_numeric(results_df[pred_col], errors='coerce')
    valid_mask = preds.notna()
    y_true = gold_bin[valid_mask]
    y_pred = preds[valid_mask].astype(int)

    print('\n' + '=' * 60)
    print(f'Modelo: {model_key}')
    print(f'Predicciones válidas: {valid_mask.sum()} / {len(results_df)}')
    print('=' * 60)
    if len(y_true) == 0:
        print('No hay predicciones válidas para este modelo.')
        continue
    print('Accuracy:', round(accuracy_score(y_true, y_pred), 4))
    print('Confusion matrix:')
    print(confusion_matrix(y_true, y_pred))
    print()
    print(classification_report(y_true, y_pred, target_names=['NO ODIO', 'ODIO'], digits=4))

Verificación del merge (primeras 5 filas):


,id,text_work,text_original,HATEFUL
0,1368339770686996480,Extranjeros participan de los destrozos realiz...,Extranjeros participan de los destrozos realiz...,0
1,1367890949792272391,@pepegoyomr @CARLOSLGUERRERO Al contrario!!! E...,@pepegoyomr @CARLOSLGUERRERO Al contrario!!! E...,0
2,1367958627739258880,La diferencia con los kirchneristas ya no es s...,La diferencia con los kirchneristas ya no es s...,0
3,1368568840351870976,@Lady_Chiyome La dictadura de pelusones del 73...,@Lady_Chiyome La dictadura de pelusones del 73...,0
4,1368382756502306816,@Mari21Llll La pinché Momia solo es un florero...,@Mari21Llll La pinché Momia solo es un florero...,0



Modelo: pysentimiento_hate
Predicciones válidas: 100 / 100
Accuracy: 0.75
Confusion matrix:
[[69 20]
 [ 5  6]]

              precision    recall  f1-score   support

     NO ODIO     0.9324    0.7753    0.8466        89
        ODIO     0.2308    0.5455    0.3243        11

    accuracy                         0.7500       100
   macro avg     0.5816    0.6604    0.5855       100
weighted avg     0.8552    0.7500    0.7892       100


Modelo: cardiff_xlm_hate_es
Predicciones válidas: 100 / 100
Accuracy: 0.84
Confusion matrix:
[[80  9]
 [ 7  4]]

              precision    recall  f1-score   support

     NO ODIO     0.9195    0.8989    0.9091        89
        ODIO     0.3077    0.3636    0.3333        11

    accuracy                         0.8400       100
   macro avg     0.6136    0.6313    0.6212       100
weighted avg     0.8522    0.8400    0.8458       100


Modelo: bne_hate_offensive_es
Predicciones válidas: 100 / 100
Accuracy: 0.57
Confusion matrix:
[[52 37]
 [ 6  5]]

   

In [22]:

from IPython.display import display, HTML

# ── 1. Preparar gold ─────────────────────────────────────────────────────────
df_gold = df_check[['id', 'text', 'HATEFUL']].rename(columns={'text': 'text_original'})

cols_work = [c for c in work_df.columns if c not in ('HATEFUL', 'text')]
results_df = (
    work_df[cols_work + ['text']].rename(columns={'text': 'text_work'})
    .merge(df_gold, on='id', how='left')
)

gold_bin = results_df['HATEFUL'].astype(int)

# ── 2. Render HTML ───────────────────────────────────────────────────────────
def render_benchmark_html(results_df, loaded_models, gold_bin):
    sections = []
    for model_key in loaded_models.keys():
        pred_col = f'{model_key}__binary_pred'
        preds = pd.to_numeric(results_df[pred_col], errors='coerce')
        valid_mask = preds.notna()
        n_valid = int(valid_mask.sum())
        y_true = gold_bin[valid_mask]
        y_pred = preds[valid_mask].astype(int)

        header = f'''<div style="margin-bottom:20px;border:1px solid #ccc;border-radius:8px;overflow:hidden;">
          <div style="background:#2c3e50;color:white;padding:10px 16px;font-size:14px;font-weight:bold;">
            {model_key}
            <span style="font-weight:normal;font-size:12px;margin-left:10px;">({n_valid}/{len(results_df)} predicciones válidas)</span>
          </div>'''

        if len(y_true) == 0:
            sections.append(header + '<div style="padding:12px;">Sin predicciones válidas.</div></div>')
            continue

        acc = accuracy_score(y_true, y_pred)
        cm = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel()
        report = classification_report(y_true, y_pred, target_names=['NO ODIO', 'ODIO'], output_dict=True)

        cm_html = f'''<div style="padding:12px 16px;">
          <b>Accuracy: {acc:.4f}</b>
          <table style="border-collapse:collapse;margin:10px 0;font-size:13px;">
            <tr>
              <td style="padding:4px 10px;background:#ecf0f1;"></td>
              <th style="padding:4px 10px;background:#ecf0f1;">Pred: NO ODIO</th>
              <th style="padding:4px 10px;background:#ecf0f1;">Pred: ODIO</th>
            </tr>
            <tr>
              <th style="padding:4px 10px;background:#ecf0f1;">Real: NO ODIO</th>
              <td style="padding:4px 10px;background:#d5f5e3;text-align:center;">{tn}</td>
              <td style="padding:4px 10px;background:#fadbd8;text-align:center;">{fp}</td>
            </tr>
            <tr>
              <th style="padding:4px 10px;background:#ecf0f1;">Real: ODIO</th>
              <td style="padding:4px 10px;background:#fadbd8;text-align:center;">{fn}</td>
              <td style="padding:4px 10px;background:#d5f5e3;text-align:center;">{tp}</td>
            </tr>
          </table>'''

        report_rows = ''
        for label in ['NO ODIO', 'ODIO', 'macro avg', 'weighted avg']:
            if label in report:
                r = report[label]
                bg = '#f8f9fa' if 'avg' in label else 'white'
                report_rows += f'''<tr style="background:{bg};">
                  <td style="padding:4px 10px;">{label}</td>
                  <td style="padding:4px 10px;text-align:right;">{r["precision"]:.4f}</td>
                  <td style="padding:4px 10px;text-align:right;">{r["recall"]:.4f}</td>
                  <td style="padding:4px 10px;text-align:right;">{r["f1-score"]:.4f}</td>
                  <td style="padding:4px 10px;text-align:right;">{int(r["support"])}</td>
                </tr>'''

        report_html = f'''<table style="border-collapse:collapse;margin:10px 0;font-size:13px;">
            <tr style="background:#ecf0f1;">
              <th style="padding:4px 10px;text-align:left;">Clase</th>
              <th style="padding:4px 10px;">Precision</th>
              <th style="padding:4px 10px;">Recall</th>
              <th style="padding:4px 10px;">F1</th>
              <th style="padding:4px 10px;">Support</th>
            </tr>
            {report_rows}
          </table></div></div>'''

        sections.append(header + cm_html + report_html)

    return '<div style="font-family:Arial,sans-serif;">' + ''.join(sections) + '</div>'

display(HTML(render_benchmark_html(results_df, loaded_models, gold_bin)))


In [23]:

import html as htmllib

# ── Configuración ─────────────────────────────────────────────────────────────
SEED = 76
N_SAMPLE = 10

sample = results_df.sample(n=min(N_SAMPLE, len(results_df)), random_state=SEED).reset_index(drop=True)
model_keys = list(loaded_models.keys())

# ── Cabecera de tabla ────────────────────────────────────────────────────────
th = lambda s: f'<th style="padding:6px 10px;white-space:nowrap;">{s}</th>'
headers = (
    th('text_work') + th('text_original') + th('match?') +
    th('HATEFUL<br>(gold)') +
    ''.join(th(mk.replace('_', '<br>')) for mk in model_keys)
)

# ── Filas ─────────────────────────────────────────────────────────────────────
def pred_cell(val):
    try:
        v = int(float(val))
        bg = '#fadbd8' if v == 1 else '#d5f5e3'
        return f'<td style="text-align:center;background:{bg};font-weight:bold;">{v}</td>'
    except:
        return '<td style="text-align:center;color:#aaa;">—</td>'

def gold_cell(val):
    v = int(val)
    bg = '#e74c3c' if v == 1 else '#27ae60'
    return f'<td style="text-align:center;background:{bg};color:white;font-weight:bold;">{v}</td>'

rows_html = ''
for _, row in sample.iterrows():
    tw = htmllib.escape(str(row.get('text_work', ''))[:180])
    to = htmllib.escape(str(row.get('text_original', ''))[:180])
    match = '✓' if str(row.get('text_work','')) == str(row.get('text_original','')) else '✗'
    match_color = 'green' if match == '✓' else 'red'

    rows_html += f'''<tr style="border-bottom:1px solid #eee;">
      <td style="padding:6px 10px;font-size:12px;max-width:220px;">{tw}</td>
      <td style="padding:6px 10px;font-size:12px;max-width:220px;">{to}</td>
      <td style="text-align:center;color:{match_color};font-weight:bold;">{match}</td>
      {gold_cell(row["HATEFUL"])}
      {"".join(pred_cell(row.get(f"{mk}__binary_pred")) for mk in model_keys)}
    </tr>'''

table_html = f'''
<div style="font-family:Arial,sans-serif;overflow-x:auto;">
  <h3 style="margin-bottom:8px;">Muestra aleatoria · seed={SEED} · n={N_SAMPLE}</h3>
  <table style="border-collapse:collapse;font-size:12px;width:100%;">
    <thead>
      <tr style="background:#2c3e50;color:white;">{headers}</tr>
    </thead>
    <tbody>{rows_html}</tbody>
  </table>
</div>'''

display(HTML(table_html))


text_work,text_original,match?,HATEFUL(gold),pysentimientohate,cardiffxlmhatees,bnehateoffensivees,betohatev2es
"@MueroPorSalir @LunaDeCafe Es verdad, si en este sexenio solo son abrazos para los delincuentes y gas lacrimógeno p… https://t.co/L8ED8M4CgY","@MueroPorSalir @LunaDeCafe Es verdad, si en este sexenio solo son abrazos para los delincuentes y gas lacrimógeno p… https://t.co/L8ED8M4CgY",✓,1,0,0,0,0
"@Duzhee_Up @Javier_1Mora @ZIMBOTANA @Pajaropolitico @lopezobrador_ @JesusRCuevas Soy madre abuela,hermana hija amig… https://t.co/3UiDSL4CVE","@Duzhee_Up @Javier_1Mora @ZIMBOTANA @Pajaropolitico @lopezobrador_ @JesusRCuevas Soy madre abuela,hermana hija amig… https://t.co/3UiDSL4CVE",✓,0,0,0,0,0
@pissanoEmi2 @alferdez @mauriciomacri Porque es un hdp! Y parece que los delincuentes que tenemos aca son pocos .,@pissanoEmi2 @alferdez @mauriciomacri Porque es un hdp! Y parece que los delincuentes que tenemos aca son pocos .,✓,0,0,0,1,0
UNA VERGUENZA El Presidente derogó el decreto de @mauriciomacri que impedía el ingreso de personas con antecedente… https://t.co/XnH5OAj6JY,UNA VERGUENZA El Presidente derogó el decreto de @mauriciomacri que impedía el ingreso de personas con antecedente… https://t.co/XnH5OAj6JY,✓,0,0,0,0,0
@Nexofin No son suficientes los delincuentes de acá que necesitan que vengan extranjeros. Destruyen todo!!,@Nexofin No son suficientes los delincuentes de acá que necesitan que vengan extranjeros. Destruyen todo!!,✓,1,1,1,1,0
@platini2008 @ElBaquedanocl @Naty_Patriota No por ser zorrones qlos dejan de ser delincuentes amigo,@platini2008 @ElBaquedanocl @Naty_Patriota No por ser zorrones qlos dejan de ser delincuentes amigo,✓,0,1,1,0,0
"@nataperol A todos los asesinos que quieren venir Argentina, espero que vayan contentos a recibirlos. Y si alguna d… https://t.co/fghAf5pZbT","@nataperol A todos los asesinos que quieren venir Argentina, espero que vayan contentos a recibirlos. Y si alguna d… https://t.co/fghAf5pZbT",✓,0,1,0,1,0
@Juan_MVilla Los procesos de revisión de un acto de expulsión son con causal objetiva. Abrir a prueba es únicamente… https://t.co/zdbiflxbSC,@Juan_MVilla Los procesos de revisión de un acto de expulsión son con causal objetiva. Abrir a prueba es únicamente… https://t.co/zdbiflxbSC,✓,0,0,0,0,0
@EldiariodeLeuco Traen delincuentes. Por votos?,@EldiariodeLeuco Traen delincuentes. Por votos?,✓,0,1,0,1,1
@calamity106 @JavierdeCaceres @TOMASMARIN67 @EmpresarioVox Estoy con usted que esto que está pasando no debería pas… https://t.co/f95sXWTATP,@calamity106 @JavierdeCaceres @TOMASMARIN67 @EmpresarioVox Estoy con usted que esto que está pasando no debería pas… https://t.co/f95sXWTATP,✓,0,0,0,0,0


In [25]:

# ── Columnas base del CSV original ───────────────────────────────────────────
LABEL_COLS = ['CALLS', 'WOMEN', 'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL']
BASE_COLS   = ['id', 'fecha', 'text'] + LABEL_COLS + ['HATEFUL']

# Filtrar solo las que existen en df_check
base_cols_present = [c for c in BASE_COLS if c in df_check.columns]

# ── Predicciones binarias de cada modelo ─────────────────────────────────────
binary_pred_cols = [f'{mk}__binary_pred' for mk in loaded_models.keys()]

# results_df tiene id + predicciones; tomamos solo esas columnas para el merge
preds_df = results_df[['id'] + binary_pred_cols].copy()

# Renombrar a nombres cortos: modelo → pred_modelo
rename_map = {f'{mk}__binary_pred': f'pred_{mk}' for mk in loaded_models.keys()}
preds_df = preds_df.rename(columns=rename_map)

# ── Merge y guardado ──────────────────────────────────────────────────────────
export_df = df_check[base_cols_present].merge(preds_df, on='id', how='left')

OUTPUT_BENCHMARK_CSV = 'outputs/benchmark_clasificaciones.csv'
export_df.to_csv(OUTPUT_BENCHMARK_CSV, index=False)

print(f"Guardado: {OUTPUT_BENCHMARK_CSV}")
print(f"Filas: {len(export_df)} | Columnas: {list(export_df.columns)}")
export_df.head(3)


Guardado: outputs/benchmark_clasificaciones.csv
Filas: 100 | Columnas: ['id', 'fecha', 'text', 'CALLS', 'WOMEN', 'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL', 'HATEFUL', 'pred_pysentimiento_hate', 'pred_cardiff_xlm_hate_es', 'pred_bne_hate_offensive_es', 'pred_beto_hate_v2_es']


,id,fecha,text,CALLS,WOMEN,LGBTI,RACISM,CLASS,POLITICS,DISABLED,APPEARANCE,CRIMINAL,HATEFUL,pred_pysentimiento_hate,pred_cardiff_xlm_hate_es,pred_bne_hate_offensive_es,pred_beto_hate_v2_es
0,1368339770686996480,2021-03-06 23:16:38+00:00,Extranjeros participan de los destrozos realiz...,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1367890949792272391,2021-03-05 17:33:11+00:00,@pepegoyomr @CARLOSLGUERRERO Al contrario!!! E...,0,0,0,0,0,0,0,0,0,0,0,0,1,0
2,1367958627739258880,2021-03-05 22:02:06+00:00,La diferencia con los kirchneristas ya no es s...,0,0,0,0,0,0,0,0,0,0,0,0,0,0
